In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [8]:
# ✅ Import libraries
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer

# ✅ Set correct model path (no .zip needed)
model_path = "/kaggle/input/huggingface/huggingface"

# ✅ Load tokenizer and model directly
tokenizer = MarianTokenizer.from_pretrained(model_path, local_files_only=True)
model = MarianMTModel.from_pretrained(model_path, local_files_only=True)


In [15]:
# ✅ Unzip the local HuggingFace model folder (no need to unzip if already a folder)
# Your model is already extracted as seen in the sidebar

# ✅ Import libraries
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer

# ✅ Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🔍 Using device:", device)

# ✅ Set local model path
model_path = "/kaggle/input/huggingface/huggingface"

# ✅ Load tokenizer and model from local path only
tokenizer = MarianTokenizer.from_pretrained(model_path, local_files_only=True)
model = MarianMTModel.from_pretrained(model_path, local_files_only=True)
model.to(device)

# ✅ Load translated dataset (first 500 rows only)
pd.read_csv("/kaggle/input/student-performance-data/translated_dataset.csv").head(500)

print("✅ Dataset loaded. Shape:", df.shape)

# ✅ Translate sample text (example)
def translate_text(text_list):
    # Tokenize text
    inputs = tokenizer(text_list, return_tensors="pt", padding=True, truncation=True).to(device)
    # Generate translations
    translated = model.generate(**inputs)
    # Decode translations
    return tokenizer.batch_decode(translated, skip_special_tokens=True)

# ✅ Try a sample from your dataframe
sample_texts = df['FATHER_OCCUPATION'].tolist()[:5]  # 🔁 Replace 'text_column_name' with actual column name
translated_texts = translate_text(sample_texts)

# ✅ Show translations
for src, tgt in zip(sample_texts, translated_texts):
    print(f"\n🔤 Original: {src}\n🌐 Translated: {tgt}")


🔍 Using device: cuda


/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


✅ Dataset loaded. Shape: (500, 81)

🔤 Original: Es operario de máquinas o conduce vehículos (taxita, chofer)
🌐 Translated: He is an operator of machines or drives vehicles (taxi, driver)

🔤 Original: Es vendedor o trabaja en atención al público
🌐 Translated: He's a salesman or works in public care.

🔤 Original: Es agricultor, pesquero o jornalero
🌐 Translated: He is a farmer, a fisherman or a day laborer.

🔤 Original: Trabaja por cuenta propia (por ejemplo plomero, electricista)
🌐 Translated: Self-employed (e.g. plumber, electrician)

🔤 Original: Tiene un trabajo de tipo auxiliar administrativo (por ejemplo, secretario o asistente)
🌐 Translated: Has a job of administrative assistant type (e.g. secretary or assistant)


In [16]:
pip install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 19.8 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ✅ Import libraries
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm  # for progress bar

# ✅ Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🔍 Using device:", device)

# ✅ Load model locally
model_path = "/kaggle/input/huggingface/huggingface"
tokenizer = MarianTokenizer.from_pretrained(model_path, local_files_only=True)
model = MarianMTModel.from_pretrained(model_path, local_files_only=True).to(device)
model.eval()  # Ensure it's in eval mode

# ✅ Use inference mode (faster)
@torch.inference_mode()
def translate_text(text_list):
    inputs = tokenizer(text_list, return_tensors="pt", padding=True, truncation=True).to(device)
    outputs = model.generate(**inputs)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# ✅ Settings
columns_to_translate = [ 'STUDENT_MUNICIPALITY_OF_RESIDENCE', 'SCHOOL_NAME', 'SCHOOL_BRANCH_NAME', 'SCHOOL_MUNICIPALITY', 'TEST_MUNICIPALITY']
chunk_size = 1000  # reduced for better memory handling
batch_size = 16    # smaller batch to reduce overload

# ✅ Load and process in chunks
translated_chunks = []
reader = pd.read_csv("/kaggle/input/dataset/translated_dataset(original).csv", chunksize=chunk_size)

for chunk in tqdm(reader, desc="🔁 Processing chunks"):
    for col in columns_to_translate:
        texts = chunk[col].fillna("").astype(str).tolist()
        translated_texts = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            translated_batch = translate_text(batch)
            translated_texts.extend(translated_batch)

        chunk[f"{col}_translated"] = translated_texts

    # 🔄 Clear GPU memory after each chunk (optional)
    torch.cuda.empty_cache()

    translated_chunks.append(chunk)

# ✅ Combine all translated chunks
final_df = pd.concat(translated_chunks, ignore_index=True)

# ✅ Save locally
output_path = "/kaggle/working/final_translated_output.csv"
final_df.to_csv(output_path, index=False)
print(f"✅ Done! Translated file saved to: {output_path}")


2025-05-27 11:35:13.934174: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748345714.133158      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748345714.186985      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🔍 Using device: cuda


/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
🔁 Processing chunks: 1it [07:37, 457.77s/it]

In [21]:
# ✅ Import libraries
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer

# ✅ Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🔍 Using device:", device)

# ✅ Load model from local Hugging Face folder
model_path = "/kaggle/input/huggingface/huggingface"
tokenizer = MarianTokenizer.from_pretrained(model_path, local_files_only=True)
model = MarianMTModel.from_pretrained(model_path, local_files_only=True).to(device)

# ✅ Define columns to translate
columns_to_translate = [
    'STUDENT_MUNICIPALITY_OF_RESIDENCE', 'SCHOOL_NAME', 'SCHOOL_BRANCH_NAME', 'SCHOOL_MUNICIPALITY', 'TEST_MUNICIPALITY'
    # ➕ Add 'col11', 'col12' if needed
]

# ✅ Define translation function
def translate_text(text_list):
    inputs = tokenizer(text_list, return_tensors="pt", padding=True, truncation=True).to(device)
    outputs = model.generate(**inputs)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# ✅ Process full dataset in chunks
chunk_size = 5000
all_translated_chunks = []

for chunk in pd.read_csv("/kaggle/input/dataset/translated_dataset(original).csv", chunksize=chunk_size):
    print(f"🔁 Processing chunk with {len(chunk)} rows...")
    
    for col in columns_to_translate:
        # Make sure column exists and convert to string
        texts = chunk[col].fillna("").astype(str).tolist()
        
        # Translate in smaller batches
        translated_texts = []
        batch_size = 32
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            translated_batch = translate_text(batch)
            translated_texts.extend(translated_batch)
        
        # Save to new column
        chunk[f"{col}_translated"] = translated_texts

    # Append processed chunk
    all_translated_chunks.append(chunk)

# ✅ Combine all chunks into a final DataFrame
final_df = pd.concat(all_translated_chunks, ignore_index=True)

# ✅ Save the final translated dataset
output_path = "/kaggle/working/final_translated_output.csv"
final_df.to_csv(output_path, index=False)
print(f"✅ All chunks translated & saved to: {output_path}")


NameError: name 'kaggle' is not defined